# 🚀 Master Pipeline Phase 2 — Advanced Upgrades (1-Click)

**Goal:** Run all Phase 2 advanced modules in a single Colab/local session:
1. **Walk-Forward CV** (5 expanding folds, OOF metrics)
2. **Optuna Hyperparameter Optimization** (maximizes OOF Sharpe with MedianPruner)
3. **Risk-Managed Backtesting** (Kelly sizing + 20% vol-targeting + -15% DD circuit breaker)
4. **SHAP Interpretability** (feature importance + high-vol vs low-vol regime comparison)

**When to use this notebook:**
- You've already run the Phase 1 Master Pipeline (or `scripts/generate_interim_features.py`) and want to apply Phase 2 upgrades.
- You want a single reproducible artifact for a recruiter to evaluate the production-grade upgrades.
- You're on Colab and want to avoid session-limit / cross-session dependency issues.

**Runtime:**
- Fast path (skip Optuna, skip SHAP): ~3 min on CPU
- Full path (Optuna 20 trials + SHAP): ~15-25 min on Colab T4 GPU, ~60-90 min on CPU

**Author:** Nassim K.


## 1. Universal Environment Setup

Auto-detects Colab vs local, clones the repo, installs deps, and sets `ROOT`.


In [ ]:
import os
import sys
from pathlib import Path

# ==========================================
# 1. ENVIRONMENT DETECTION & AUTO-SETUP
# ==========================================
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    REPO_URL = "https://github.com/nassim0014/btc-llm-sentiment.git"
    REPO_NAME = "btc-llm-sentiment"
    COLAB_ROOT = Path('/content') / REPO_NAME
    if not COLAB_ROOT.exists():
        print(f"🚀 Colab environment detected. Cloning repository...")
        !git clone {REPO_URL} /content/{REPO_NAME}
        print("📦 Installing dependencies...")
        !pip install -q -r /content/{REPO_NAME}/requirements.txt
    else:
        print(f"✅ Repository already exists in Colab.")
    ROOT = COLAB_ROOT
else:
    current_dir = Path.cwd()
    ROOT = None
    for _ in range(6):
        if (current_dir / 'requirements.txt').exists():
            ROOT = current_dir
            break
        if current_dir == current_dir.parent:
            break
        current_dir = current_dir.parent
    if ROOT is None:
        raise FileNotFoundError(
            "❌ Could not locate the project root (missing 'requirements.txt').\n"
            "If running locally, please ensure you have cloned the repo and opened this notebook from within the project directory."
        )

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(f"✅ Environment initialized. Root directory: {ROOT}")

---
## 2. Pipeline Configuration

Set the flags below to control which Phase 2 modules run.

| Flag | Default | Effect |
|------|---------|--------|
| `RUN_WALK_FORWARD` | `True` | Run baseline walk-forward CV (5 folds) |
| `RUN_OPTUNA` | `True` | Run Optuna HPO with walk-forward objective |
| `N_OPTUNA_TRIALS` | `20` | Number of Optuna trials (each = 5 fold trainings) |
| `RUN_RISK_MANAGED` | `True` | Run risk-managed backtest with best params |
| `USE_SHAP` | `True` | Run SHAP interpretability + regime comparison |
| `USE_PRECOMPUTED_FEATURES` | `True` | Load existing feature bundle if available; rebuild inline if missing |


In [ ]:
# ========================================================
# CONFIGURATION — change these flags to control the pipeline
# ========================================================
RUN_WALK_FORWARD      = True    # Stage 3: baseline walk-forward CV
RUN_OPTUNA            = True    # Stage 4: Optuna HPO
N_OPTUNA_TRIALS       = 20      # only used if RUN_OPTUNA = True
OPTUNA_EPOCHS         = 10      # per-fold training epochs during search
RUN_RISK_MANAGED      = True    # Stage 5: risk-managed backtest
USE_SHAP              = True    # Stage 6: SHAP interpretability
USE_PRECOMPUTED_FEATURES = True # load existing bundle or rebuild inline

import numpy as np
import pandas as pd

INTERIM = ROOT / 'notebooks' / 'interim'
OUTPUTS = ROOT / 'outputs'
INTERIM.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print(f'ROOT     : {ROOT}')
print(f'INTERIM  : {INTERIM}')
print(f'OUTPUTS  : {OUTPUTS}')
print(f'Flags    : WF={RUN_WALK_FORWARD}  Optuna={RUN_OPTUNA}  Risk={RUN_RISK_MANAGED}  SHAP={USE_SHAP}')

---
## 3. Stage 1 — Feature Bundle (load or rebuild inline)

Loads `notebooks/interim/features_for_lstm.pkl` if it exists (produced by the Phase 1 Master Pipeline or `scripts/generate_interim_features.py`). If missing, rebuilds it inline from `Data/cryptonews.csv` + yfinance.


In [ ]:
import pickle
import os

FEATURE_BUNDLE = INTERIM / 'features_for_lstm.pkl'
MERGED_PARQUET = INTERIM / 'merged_with_llm_sentiment.parquet'

print('--- Stage 1: Feature Bundle ---')
if USE_PRECOMPUTED_FEATURES and FEATURE_BUNDLE.exists():
    with FEATURE_BUNDLE.open('rb') as f:
        bundle = pickle.load(f)
    print(f'  Loaded existing bundle: {FEATURE_BUNDLE}')
    print(f'  Train: {bundle["train_x"].shape}  Val: {bundle["val_x"].shape}  Test: {bundle["test_x"].shape}')
else:
    print('  Feature bundle not found — rebuilding inline ...')
    import yfinance as yf
    import ast
    from io import StringIO
    import requests
    from sklearn.preprocessing import StandardScaler
    HORIZON = 5

    # 1. News
    NEWS_URL = 'https://raw.githubusercontent.com/nassim0014/btc-llm-sentiment/main/Data/cryptonews.csv'
    LOCAL_NEWS = ROOT / 'Data' / 'cryptonews.csv'
    try:
        r = requests.get(NEWS_URL, timeout=120); r.raise_for_status()
        news = pd.read_csv(StringIO(r.text))
        print(f'  News: {len(news):,} rows (remote)')
    except Exception as e:
        news = pd.read_csv(LOCAL_NEWS)
        print(f'  News: {len(news):,} rows (local fallback)')
    news['date'] = pd.to_datetime(news['date'], format='mixed', utc=True, errors='coerce')
    news = news.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
    def parse(s):
        try:
            d = ast.literal_eval(s) if isinstance(s, str) else {}
            return pd.Series({'sentiment_class': d.get('class','neutral'),
                              'sentiment_polarity': float(d.get('polarity',0.0)),
                              'sentiment_subjectivity': float(d.get('subjectivity',0.0))})
        except Exception:
            return pd.Series({'sentiment_class':'neutral','sentiment_polarity':0.0,'sentiment_subjectivity':0.0})
    sent = news['sentiment'].apply(parse)
    news = pd.concat([news.drop(columns=['sentiment']), sent], axis=1)
    news['llm_sentiment'] = news['sentiment_polarity'].astype(float)

    # 2. BTC
    btc = yf.download('BTC-USD', start='2023-01-01', end='2024-12-31', auto_adjust=False, progress=False)
    if isinstance(btc.columns, pd.MultiIndex):
        btc.columns = [' '.join(c).strip() for c in btc.columns]
    btc = btc.reset_index().rename(columns={'Date':'date'})
    btc.columns = [c.replace(' BTC-USD','').lower() for c in btc.columns]
    btc['date'] = pd.to_datetime(btc['date']).dt.floor('D')

    # 3. Merge + features
    news['date_day'] = news['date'].dt.tz_convert(None).dt.floor('D')
    daily_news = (news.groupby('date_day')
        .agg(news_count=('title','size'),
             mean_polarity=('sentiment_polarity','mean'),
             mean_subjectivity=('sentiment_subjectivity','mean'),
             neg_share=('sentiment_class', lambda s: (s=='negative').mean()),
             pos_share=('sentiment_class', lambda s: (s=='positive').mean()),
             llm_sentiment_mean=('llm_sentiment','mean'),
             llm_sentiment_std=('llm_sentiment','std'),
             llm_headline_count=('llm_sentiment','size'),
             llm_pos_share=('llm_sentiment', lambda s: (s>0.3).mean()),
             llm_neg_share=('llm_sentiment', lambda s: (s<-0.3).mean()))
        .reset_index().rename(columns={'date_day':'date'})
        .fillna({'llm_sentiment_std':0}))
    df = pd.merge(btc, daily_news, on='date', how='left')
    fill_cols = ['news_count','mean_polarity','neg_share','pos_share',
                 'llm_sentiment_mean','llm_sentiment_std','llm_headline_count',
                 'llm_pos_share','llm_neg_share']
    df[fill_cols] = df[fill_cols].fillna(0)
    df = df.sort_values('date').reset_index(drop=True)

    def rsi(close, period=14):
        delta = close.diff()
        gain = delta.clip(lower=0).ewm(alpha=1/period, adjust=False).mean()
        loss = (-delta.clip(upper=0)).ewm(alpha=1/period, adjust=False).mean()
        return 100 - (100 / (1 + gain/(loss+1e-12)))
    df['ret_1d'] = np.log(df['close']/df['close'].shift(1))
    df['ret_3d'] = np.log(df['close']/df['close'].shift(3))
    df['ret_7d'] = np.log(df['close']/df['close'].shift(7))
    df['vol_7d'] = df['ret_1d'].rolling(7).std()
    df['vol_21d'] = df['ret_1d'].rolling(21).std()
    df['rsi_14'] = rsi(df['close'], 14)
    ema_fast = df['close'].ewm(span=12, adjust=False).mean()
    ema_slow = df['close'].ewm(span=26, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    df['macd_line'] = macd_line
    df['macd_hist'] = macd_line - macd_line.ewm(span=9, adjust=False).mean()
    bb_mid = df['close'].rolling(20).mean()
    bb_std = df['close'].rolling(20).std()
    df['bb_pct_b'] = (df['close'] - (bb_mid - 2*bb_std)) / (4*bb_std + 1e-12)
    df['bb_width'] = (4*bb_std) / (bb_mid + 1e-12)
    for lag in [1,2,3,5]:
        df[f'llm_sent_lag{lag}'] = df['llm_sentiment_mean'].shift(lag)
        df[f'llm_pos_share_lag{lag}'] = df['llm_pos_share'].shift(lag)
    df['llm_sent_3d_ma'] = df['llm_sentiment_mean'].rolling(3).mean().shift(1)
    df['llm_sent_5d_ma'] = df['llm_sentiment_mean'].rolling(5).mean().shift(1)
    df['forward_ret_5d'] = df['close'].shift(-HORIZON)/df['close'] - 1
    df['target_up_5d'] = (df['forward_ret_5d'] > 0).astype(int)
    df = df.dropna(subset=['target_up_5d']).reset_index(drop=True)

    FEATURE_COLS = ['ret_1d','ret_3d','ret_7d','vol_7d','vol_21d',
                    'rsi_14','macd_line','macd_hist','bb_pct_b','bb_width',
                    'llm_sent_lag1','llm_sent_lag2','llm_sent_lag3','llm_sent_lag5',
                    'llm_pos_share_lag1','llm_pos_share_lag3','llm_pos_share_lag5',
                    'llm_sent_3d_ma','llm_sent_5d_ma',
                    'news_count','mean_polarity','neg_share','pos_share']
    n = len(df); n_train = int(n*0.70); n_val = int(n*0.15)
    train, val, test = df.iloc[:n_train], df.iloc[n_train:n_train+n_val], df.iloc[n_train+n_val:]
    scaler = StandardScaler()
    train_x = scaler.fit_transform(train[FEATURE_COLS].fillna(0)).reshape(-1,1,len(FEATURE_COLS))
    val_x = scaler.transform(val[FEATURE_COLS].fillna(0)).reshape(-1,1,len(FEATURE_COLS))
    test_x = scaler.transform(test[FEATURE_COLS].fillna(0)).reshape(-1,1,len(FEATURE_COLS))
    bundle = dict(
        train_x=train_x, train_y=train['target_up_5d'].values,
        val_x=val_x, val_y=val['target_up_5d'].values,
        test_x=test_x, test_y=test['target_up_5d'].values,
        feature_cols=FEATURE_COLS, scaler=scaler,
        train_close=train['close'].values, val_close=val['close'].values,
        test_close=test['close'].values,
        train_dates=train['date'].values, val_dates=val['date'].values,
        test_dates=test['date'].values,
        test_forward_ret_5d=test['forward_ret_5d'].values,
    )
    df.to_parquet(MERGED_PARQUET, index=False)
    with FEATURE_BUNDLE.open('wb') as f:
        pickle.dump(bundle, f)
    print(f'  Rebuilt and saved: {FEATURE_BUNDLE}')
    print(f'  Train: {train_x.shape}  Val: {val_x.shape}  Test: {test_x.shape}')

# Reconstruct full-window arrays for walk-forward CV
X = np.concatenate([bundle['train_x'], bundle['val_x'], bundle['test_x']], axis=0)
y = np.concatenate([bundle['train_y'], bundle['val_y'], bundle['test_y']], axis=0)
dates = np.concatenate([bundle['train_dates'], bundle['val_dates'], bundle['test_dates']], axis=0)
n_features = X.shape[-1]

# Load merged parquet for close prices across the full window
if MERGED_PARQUET.exists():
    merged = pd.read_parquet(MERGED_PARQUET).sort_values('date').reset_index(drop=True)
    close = merged['close'].values
else:
    # Fallback: only test_close is in the bundle; use it for the test window
    close = np.concatenate([bundle['train_close'], bundle['val_close'], bundle['test_close']], axis=0)

print(f'  Full window: X={X.shape}  y={y.shape}  close={close.shape}')
print(f'  Date range: {pd.Timestamp(dates[0]).date()} → {pd.Timestamp(dates[-1]).date()}')
print('✅ Stage 1 complete.')

---
## 4. Stage 2 — Walk-Forward Cross-Validation (Baseline)

Runs 5 expanding-window folds with a baseline LSTM (lr=1e-3, units=64, dropout=0, layers=1). Logs per-fold OOF metrics (Sharpe, Accuracy, F1, AUC, Max DD). This establishes the regime-stability baseline that Optuna will try to beat.


In [ ]:
if not RUN_WALK_FORWARD:
    print('⏭️  Skipping walk-forward CV (RUN_WALK_FORWARD = False)')
else:
    import os
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    import tensorflow as tf
    from tensorflow.keras import layers, models, callbacks
    from sklearn.utils.class_weight import compute_class_weight
    from src.cv.walk_forward import walk_forward_splits, evaluate_oof_metrics
    tf.get_logger().setLevel('ERROR')
    tf.random.set_seed(RANDOM_STATE)

    print('--- Stage 2: Walk-Forward CV (Baseline) ---')

    # Class weights
    classes = np.unique(y)
    weights = compute_class_weight('balanced', classes=classes, y=y)
    class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
    print(f'  Class weights: {class_weight}')

    # Baseline model factory
    def build_baseline():
        inp = layers.Input(shape=(1, n_features), name='features')
        x = layers.LSTM(64, return_sequences=False)(inp)
        x = layers.Dense(16, activation='relu')(x)
        out = layers.Dense(1, activation='sigmoid', name='prob_up')(x)
        m = models.Model(inp, out, name='baseline_lstm_wf')
        m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
        return m

    fold_metrics = []
    n = len(X)
    for fold in walk_forward_splits(n=n, n_folds=5, min_train=400, val_size=60):
        X_tr, y_tr = X[fold.train_idx], y[fold.train_idx]
        X_va, y_va = X[fold.val_idx], y[fold.val_idx]
        close_va = close[fold.val_idx]
        dates_va = dates[fold.val_idx]

        tf.keras.backend.clear_session()
        tf.random.set_seed(RANDOM_STATE)
        model = build_baseline()
        model.fit(X_tr, y_tr, validation_data=(X_va, y_va),
                  epochs=20, batch_size=32, verbose=0,
                  class_weight=class_weight,
                  callbacks=[callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True),
                             callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5)])
        y_prob_va = model.predict(X_va, verbose=0).ravel()
        m = evaluate_oof_metrics(y_true=y_va, y_prob=y_prob_va, close=close_va, threshold=0.5, fee=0.001)
        m['fold'] = fold.fold_num
        m['val_start'] = pd.Timestamp(dates_va[0]).isoformat() if len(dates_va) else ''
        m['val_end'] = pd.Timestamp(dates_va[-1]).isoformat() if len(dates_va) else ''
        fold_metrics.append(m)
        print(f'  Fold {fold.fold_num}: acc={m["accuracy"]:.3f}  f1={m["f1"]:.3f}  auc={m["auc"]:.3f}  sharpe={m["sharpe"]:+.3f}  max_dd={m["max_dd"]:+.3f}  trades={m["n_trades"]}')

    wf_df = pd.DataFrame(fold_metrics)
    wf_df.to_csv(OUTPUTS / 'walk_forward_oof_metrics.csv', index=False)
    print(f'\n  OOF Sharpe   : {wf_df.sharpe.mean():+.3f} ± {wf_df.sharpe.std():.3f}')
    print(f'  OOF Accuracy : {wf_df.accuracy.mean():.3f} ± {wf_df.accuracy.std():.3f}')
    print(f'  OOF F1       : {wf_df.f1.mean():.3f} ± {wf_df.f1.std():.3f}')
    print(f'  Saved → {OUTPUTS / "walk_forward_oof_metrics.csv"}')
    print('✅ Stage 2 complete.')

### 2.1 Memory Cleanup After Walk-Forward CV

Release the 5 trained models from the TF graph before starting Optuna.


In [ ]:
import gc
try:
    import tensorflow as tf
    tf.keras.backend.clear_session()
except Exception:
    pass
try:
    del model
except NameError:
    pass
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except ImportError:
    pass
print('✅ Memory released after walk-forward CV.')

---
## 5. Stage 3 — Optuna Hyperparameter Optimization

Search space: `lr` (log-uniform 1e-4 to 1e-2), `units` (32/64/128), `dropout` (0.0/0.2/0.4), `num_layers` (1/2).

Objective: maximize mean OOF Sharpe across 5 walk-forward folds. MedianPruner kills underperforming trials after fold 2.


In [ ]:
if not RUN_OPTUNA:
    print('⏭️  Skipping Optuna search (RUN_OPTUNA = False)')
    # Try to load existing best params
    import json
    best_params_path = OUTPUTS / 'best_optuna_params.json'
    if best_params_path.exists():
        with best_params_path.open() as f:
            best_hp = json.load(f)['best_params']
        print(f'  Loaded existing best params: {best_hp}')
    else:
        best_hp = {'lr': 1e-3, 'units': 64, 'dropout': 0.0, 'num_layers': 1}
        print(f'  Using default params: {best_hp}')
else:
    import os
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    from src.cv.optuna_search import run_optuna_search
    from sklearn.utils.class_weight import compute_class_weight
    import tensorflow as tf
    tf.get_logger().setLevel('ERROR')
    tf.random.set_seed(RANDOM_STATE)

    print(f'--- Stage 3: Optuna HPO ({N_OPTUNA_TRIALS} trials × 5 folds × {OPTUNA_EPOCHS} epochs) ---')

    classes = np.unique(y)
    weights = compute_class_weight('balanced', classes=classes, y=y)
    class_weight = {int(c): float(w) for c, w in zip(classes, weights)}

    best_hp = run_optuna_search(
        X=X, y=y, close=close, dates=dates,
        n_features=n_features,
        n_trials=N_OPTUNA_TRIALS,
        n_folds=5, min_train=400, val_size=60,
        epochs=OPTUNA_EPOCHS, batch_size=32,
        class_weight=class_weight, fee=0.001, threshold=0.5,
        pruner_n_startup=3, pruner_n_warmup=2,
        output_path=OUTPUTS / 'best_optuna_params.json',
        seed=RANDOM_STATE,
    )
    print(f'\n  Best params: {best_hp}')
    print('✅ Stage 3 complete.')

### 3.1 Memory Cleanup After Optuna

Optuna trains many models; clear the TF session before the final model training.


In [ ]:
import gc
try:
    import tensorflow as tf
    tf.keras.backend.clear_session()
except Exception:
    pass
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except ImportError:
    pass
print('✅ Memory released after Optuna search.')

---
## 6. Stage 4 — Risk-Managed Backtesting

Trains the final LSTM with the best Optuna params on train+val, then runs the risk-managed backtest (Kelly fraction sizing + 20% vol-targeting + -15% drawdown circuit breaker) on the test window.


In [ ]:
if not RUN_RISK_MANAGED:
    print('⏭️  Skipping risk-managed backtest (RUN_RISK_MANAGED = False)')
else:
    import os
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    import tensorflow as tf
    from tensorflow.keras import layers, models, callbacks, regularizers
    from sklearn.utils.class_weight import compute_class_weight
    from src.backtest.risk_managed import risk_managed_backtest, compare_strategies
    tf.get_logger().setLevel('ERROR')
    tf.random.set_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)

    print('--- Stage 4: Risk-Managed Backtest ---')
    print(f'  Best params: {best_hp}')

    train_x, train_y = bundle['train_x'], bundle['train_y']
    val_x, val_y = bundle['val_x'], bundle['val_y']
    test_x, test_y = bundle['test_x'], bundle['test_y']
    test_close = bundle['test_close']
    test_dates = pd.to_datetime(bundle['test_dates'])

    X_full = np.concatenate([train_x, val_x], axis=0)
    y_full = np.concatenate([train_y, val_y], axis=0)
    classes = np.unique(y_full)
    weights = compute_class_weight('balanced', classes=classes, y=y_full)
    class_weight = {int(c): float(w) for c, w in zip(classes, weights)}

    # Build final model with best params
    inp = layers.Input(shape=(1, n_features), name='features')
    x = inp
    for i in range(best_hp['num_layers']):
        return_seq = (i < best_hp['num_layers'] - 1)
        x = layers.LSTM(best_hp['units'], return_sequences=return_seq,
                        dropout=best_hp['dropout'], recurrent_dropout=best_hp['dropout'])(x)
    x = layers.Dense(16, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inp, out, name='final_optuna_lstm')
    model.compile(optimizer=tf.keras.optimizers.Adam(best_hp['lr']),
                  loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

    print(f'  Training final model on {len(X_full)} samples (train+val) ...')
    history = model.fit(
        X_full, y_full, validation_split=0.15,
        epochs=30, batch_size=32, verbose=0,
        class_weight=class_weight,
        callbacks=[callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=7, restore_best_weights=True),
                   callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)],
    )
    print(f'  Trained {len(history.history["loss"])} epochs')

    test_prob = model.predict(test_x, verbose=0).ravel()
    print(f'  Test predictions: mean={test_prob.mean():.3f} std={test_prob.std():.3f}')

    # Save model for SHAP
    model.save(INTERIM / 'best_optuna_model.keras')
    print(f'  Saved model → {INTERIM / "best_optuna_model.keras"}')

    # Risk-managed backtest
    rm = risk_managed_backtest(
        prob=test_prob, close=test_close,
        threshold=0.5, fee=0.001,
        target_annual_vol=0.20, vol_lookback=20,
        max_drawdown_pct=0.15,
    )
    summary = rm.to_summary_dict()
    print(f'\n  Risk-Managed Backtest Results:')
    for k, v in summary.items():
        print(f'    {k:30s} {v}')

    # Strategy comparison
    comparison = compare_strategies(test_prob, test_close, threshold=0.5, fee=0.001)
    print(f'\n  Strategy Comparison:')
    print(comparison.to_string(index=False))

    # Save outputs
    pd.DataFrame([summary]).to_csv(OUTPUTS / 'risk_managed_backtest_results.csv', index=False)
    comparison.to_csv(OUTPUTS / 'strategy_comparison.csv', index=False)
    equity_df = pd.DataFrame({
        'date': test_dates, 'equity': rm.equity, 'position': rm.positions,
        'kelly_fraction': rm.kelly_fraction, 'vol_target_factor': rm.vol_target_factor,
        'raw_prob': rm.raw_signal,
    })
    equity_df.to_csv(OUTPUTS / 'risk_managed_equity_curve.csv', index=False)
    print(f'\n  Saved → outputs/risk_managed_backtest_results.csv')
    print(f'  Saved → outputs/strategy_comparison.csv')
    print(f'  Saved → outputs/risk_managed_equity_curve.csv')
    print('✅ Stage 4 complete.')

### 4.1 Memory Cleanup After Backtest

Release the final LSTM before SHAP (SHAP will reload the model from disk).


In [ ]:
import gc
try:
    import tensorflow as tf
    tf.keras.backend.clear_session()
except Exception:
    pass
try:
    del model, history
except NameError:
    pass
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except ImportError:
    pass
print('✅ Memory released after risk-managed backtest.')

---
## 7. Stage 5 — SHAP Interpretability

Computes SHAP values for the Optuna-tuned model on the test set. Tries DeepExplainer → GradientExplainer → KernelExplainer (KernelExplainer is used on TF 2.21 due to a gradient-registry incompatibility).

Generates:
- `outputs/shap_summary.png` — global beeswarm plot
- `outputs/shap_regime_comparison.png` — high-vol vs low-vol regime comparison
- `outputs/shap_feature_importance.csv` — mean |SHAP| per feature


In [ ]:
if not USE_SHAP:
    print('⏭️  Skipping SHAP (USE_SHAP = False)')
else:
    import os
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    import tensorflow as tf
    from src.interpretability.shap_explainer import run_shap_analysis
    tf.get_logger().setLevel('ERROR')

    print('--- Stage 5: SHAP Interpretability ---')

    # Load the saved model
    model_path = INTERIM / 'best_optuna_model.keras'
    if not model_path.exists():
        print(f'  ⚠️  Model not found at {model_path} — skipping SHAP.')
        print(f'      Run Stage 4 (Risk-Managed Backtest) first to train and save the model.')
    else:
        model = tf.keras.models.load_model(model_path)
        print(f'  Loaded model: {model.name}')

        train_x = bundle['train_x']
        test_x = bundle['test_x']
        test_close = bundle['test_close']
        feature_names = bundle['feature_cols']

        result = run_shap_analysis(
            model=model,
            train_x=train_x,
            test_x=test_x,
            feature_names=feature_names,
            test_close=test_close,
            output_dir=OUTPUTS,
        )
        print(f'\n  Explainer used: {result["explainer_used"]}')
        print(f'  SHAP values shape: {result["shap_values_shape"]}')
        print(f'  Regime split: {result["regime_info"]}')
        print('✅ Stage 5 complete.')

### 5.1 Final Memory Cleanup

Release all remaining model + SHAP objects so the session can be safely closed or reused.


In [ ]:
import gc
try:
    import tensorflow as tf
    tf.keras.backend.clear_session()
except Exception:
    pass
try:
    del model
except NameError:
    pass
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'CUDA cache cleared. Allocated: {torch.cuda.memory_allocated()/1e6:.1f} MB')
except ImportError:
    pass
print('✅ All memory released. Safe to close the session.')

---
## 8. Pipeline Complete

All Phase 2 modules executed in a single session. Outputs in `outputs/`:
- `walk_forward_oof_metrics.csv` — per-fold OOF Sharpe/Accuracy/F1 (Stage 2)
- `best_optuna_params.json` — best hyperparameters from Optuna (Stage 3)
- `risk_managed_backtest_results.csv` — Kelly + vol-target + DD breaker metrics (Stage 4)
- `strategy_comparison.csv` — Simple vs Risk-Managed vs Buy & Hold (Stage 4)
- `risk_managed_equity_curve.csv` — daily equity + positions (Stage 4)
- `shap_summary.png` — global SHAP beeswarm (Stage 5)
- `shap_regime_comparison.png` — high-vol vs low-vol regimes (Stage 5)
- `shap_feature_importance.csv` — mean |SHAP| per feature (Stage 5)

### Memory management recap
- After Stage 2 (Walk-Forward CV): `tf.keras.backend.clear_session()` + `gc.collect()`
- After Stage 3 (Optuna): `tf.keras.backend.clear_session()` + `gc.collect()` + `torch.cuda.empty_cache()`
- After Stage 4 (Backtest): `del model, history` + `clear_session()` + `gc.collect()`
- After Stage 5 (SHAP): final cleanup — safe to close session

### Safe to close the session now
All artifacts are persisted to `outputs/` and `notebooks/interim/`. Commit + push to make them permanent.


In [ ]:
print('🎉 Master Pipeline Phase 2 complete!')
print(f'\nOutputs in {OUTPUTS}:')
for f in sorted(OUTPUTS.iterdir()):
    print(f'  {f.name:40s}  {f.stat().st_size / 1024:.1f} KB')
print(f'\nInterim artifacts in {INTERIM}:')
for f in sorted(INTERIM.iterdir()):
    if f.is_file():
        print(f'  {f.name:40s}  {f.stat().st_size / 1024:.1f} KB')